# Cricket Shot Classification - 3D CNN Training

This notebook trains a 3D Convolutional Neural Network to classify
cricket batting videos into 10 different shot categories.

Each video is represented using 16 uniformly sampled frames resized
to 112 × 112 pixels.

In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import numpy as np
import matplotlib.pyplot as plt

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.20.0


In [2]:
# Video preprocessing configuration

NUM_FRAMES = 16
FRAME_HEIGHT = 112
FRAME_WIDTH = 112
CHANNELS = 3

NUM_CLASSES = 10

CLASS_NAMES = [
    "cover",
    "defense",
    "flick",
    "hook",
    "late_cut",
    "lofted",
    "pull",
    "square_cut",
    "straight",
    "sweep"
]

INPUT_SHAPE = (
    NUM_FRAMES,
    FRAME_HEIGHT,
    FRAME_WIDTH,
    CHANNELS
)

print("Model input shape:", INPUT_SHAPE)
print("Number of classes:", NUM_CLASSES)
print("Classes:", CLASS_NAMES)

Model input shape: (16, 112, 112, 3)
Number of classes: 10
Classes: ['cover', 'defense', 'flick', 'hook', 'late_cut', 'lofted', 'pull', 'square_cut', 'straight', 'sweep']


In [3]:
model = keras.Sequential([
    layers.Input(shape=INPUT_SHAPE),

    layers.Conv3D(
        filters=32,
        kernel_size=(3, 3, 3),
        activation="relu",
        padding="same"
    ),

    layers.MaxPooling3D(
        pool_size=(1, 2, 2)
    ),

    layers.Conv3D(
        filters=64,
        kernel_size=(3, 3, 3),
        activation="relu",
        padding="same"
    ),

    layers.MaxPooling3D(
        pool_size=(2, 2, 2)
    ),

    layers.Conv3D(
        filters=128,
        kernel_size=(3, 3, 3),
        activation="relu",
        padding="same"
    ),

    layers.MaxPooling3D(
        pool_size=(2, 2, 2)
    ),

    layers.GlobalAveragePooling3D(),

    layers.Dropout(0.5),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dropout(0.3),

    layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

In [4]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv3d (Conv3D)                 │ (None, 16, 112, 112,   │         2,624 │
│                                 │ 32)                    │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d (MaxPooling3D)    │ (None, 16, 56, 56, 32) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_1 (Conv3D)               │ (None, 16, 56, 56, 64) │        55,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_1 (MaxPooling3D)  │ (None, 8, 28, 28, 64)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_2 (Conv3D)               │ (None, 8, 28, 28, 128) │       221,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_2 (MaxPooling3D)  │ (None, 4, 14, 14, 128) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling3d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling3D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 297,098 (1.13 MB)

 Trainable params: 297,098 (1.13 MB)

 Non-trainable params: 0 (0.00 B)

## Model Compilation

The model is compiled using the Adam optimizer and sparse categorical
cross-entropy loss for multi-class cricket shot classification.
Accuracy is used as the primary training metric.

In [5]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")

Model compiled successfully.
